In [ ]:
import pandas as pd

def transform_water_quality_data(input_file, output_file):
    """
    Transform water quality data from horizontal to vertical format based on all variables.
    Handles duplicate timestamps by using sample ID in the index.

    Parameters:
    input_file (str): Path to input CSV file
    output_file (str): Path to output CSV file
    """
    # Read the CSV file with low_memory=False to handle mixed types
    df = pd.read_csv(input_file, low_memory=False)

    # Convert to datetime
    df['DATE_TIME_HEURE'] = pd.to_datetime(df['DATE_TIME_HEURE'])

    # Convert VALUE_VALEUR to float, handling any non-numeric values
    df['VALUE_VALEUR'] = pd.to_numeric(df['VALUE_VALEUR'], errors='coerce')

    # Create a unique index using both date and sample ID
    df_pivoted = df.pivot_table(
        index=['DATE_TIME_HEURE', 'SAMPLE_ID_ÉCHANTILLON'],
        columns='VARIABLE',
        values='VALUE_VALEUR',
        aggfunc='first'  # In case of duplicates, take the first value
    )

    # Reset index to make DATE_TIME_HEURE and SAMPLE_ID_ÉCHANTILLON columns
    df_pivoted.reset_index(inplace=True)

    # Sort by date
    df_pivoted.sort_values('DATE_TIME_HEURE', inplace=True)

    # Save to CSV
    df_pivoted.to_csv(output_file, index=False)

    # Print information about the transformation
    print(f"Transformation complete!")
    print(f"Input shape: {df.shape}")
    print(f"Output shape: {df_pivoted.shape}")
    print(f"Total variables: {len(df_pivoted.columns) - 2}")  # -2 to exclude DATE_TIME_HEURE and SAMPLE_ID
    print("\nFirst few columns:", list(df_pivoted.columns[:5]))
    print("\nSample of missing data percentage for first 5 variables:")
    print(df_pivoted.isnull().mean().head())

# Usage example
if __name__ == "__main__":
    input_file = "Water-Qual-Eau-Churchill-2000-present.csv"
    output_file = "transformed_water_quality.csv"
    transform_water_quality_data(input_file, output_file)

Transformation complete!
Input shape: (72682, 12)
Output shape: (645, 275)
Total variables: 273

First few columns: ['DATE_TIME_HEURE', 'SAMPLE_ID_ÉCHANTILLON', '1,2,4,5-TETRABROMOBENZENE (1,2,4,5-TBB)', '1,3,5-TRIBROMOBENZENE', '1,3-DIBROMOBENZENE']

Sample of missing data percentage for first 5 variables:
VARIABLE
DATE_TIME_HEURE                            0.000000
SAMPLE_ID_ÉCHANTILLON                      0.000000
1,2,4,5-TETRABROMOBENZENE (1,2,4,5-TBB)    0.897674
1,3,5-TRIBROMOBENZENE                      0.986047
1,3-DIBROMOBENZENE                         0.986047
dtype: float64


In [1]:
import pandas as pd
import numpy as np

def transform_gaspe_data(input_file, output_file):
    """
    Transform Gaspé water quality data from horizontal to vertical format.

    Parameters:
    input_file (str): Path to input CSV file
    output_file (str): Path to output CSV file
    """
    print("Reading CSV file...")
    # Read the CSV file with proper handling of mixed types
    df = pd.read_csv(input_file, low_memory=False)

    # Convert to datetime
    df['DATE_TIME_HEURE'] = pd.to_datetime(df['DATE_TIME_HEURE'])

    print("Creating pivot table...")
    # Create pivot table with multiple index columns to handle duplicates
    df_pivoted = df.pivot_table(
        index=['DATE_TIME_HEURE', 'SAMPLE_ID_ÉCHANTILLON', 'SITE_NO'],
        columns='VARIABLE',
        values='VALUE_VALEUR',
        aggfunc='first'  # Take first value if duplicates exist
    )

    # Reset index to make the index columns regular columns
    df_pivoted.reset_index(inplace=True)

    # Sort by date
    df_pivoted.sort_values('DATE_TIME_HEURE', inplace=True)

    print("Saving transformed data...")
    # Save to CSV
    df_pivoted.to_csv(output_file, index=False)

    # Calculate and print summary statistics
    print("\nTransformation Summary:")
    print(f"Original number of rows: {len(df)}")
    print(f"Number of unique sample IDs: {len(df['SAMPLE_ID_ÉCHANTILLON'].unique())}")
    print(f"Transformed number of rows: {len(df_pivoted)}")
    print(f"Number of variables (columns): {len(df_pivoted.columns) - 3}")  # -3 for index columns
    print(f"Number of unique sites: {len(df_pivoted['SITE_NO'].unique())}")
    print(f"Date range: {df_pivoted['DATE_TIME_HEURE'].min()} to {df_pivoted['DATE_TIME_HEURE'].max()}")

    # Calculate value counts for each site
    site_counts = df_pivoted['SITE_NO'].value_counts()
    print("\nSamples per site:")
    print(site_counts)

    # Calculate percentage of missing values for each variable
    missing_percentages = (df_pivoted.isnull().sum() / len(df_pivoted) * 100).round(2)
    print("\nTop 10 most frequently measured variables (excluding identifiers):")
    variable_cols = [col for col in missing_percentages.index if col not in ['DATE_TIME_HEURE', 'SAMPLE_ID_ÉCHANTILLON', 'SITE_NO']]
    most_complete = missing_percentages[variable_cols].sort_values()[:10]
    print(most_complete)

    print(f"\nOutput saved to: {output_file}")

if __name__ == "__main__":
    input_file = "/content/Water-Qual-Eau-Pacific-Coastal-Cote-Pacifique-2000-present.csv"
    output_file = "Eau-Pacific-Coastal-Cote-Pacifique.csv"
    transform_gaspe_data(input_file, output_file)

Reading CSV file...
Creating pivot table...
Saving transformed data...

Transformation Summary:
Original number of rows: 274196
Number of unique sample IDs: 4352
Transformed number of rows: 4352
Number of variables (columns): 227
Number of unique sites: 13
Date range: 2000-01-02 14:30:00 to 2024-10-21 08:45:00

Samples per site:
SITE_NO
BC08HD0004    617
BC08HA0018    608
BC08HB0019    508
BC08EF0001    465
AK08DC0001    445
YT08AA0010    425
BC08GA0010    405
BC08HB0018    316
BC08FC0001    168
BC08EE0009    126
BC08CG0001    121
YT08AB0009    118
BC08HB0027     30
Name: count, dtype: int64

Top 10 most frequently measured variables (excluding identifiers):
VARIABLE
TURBIDITY               1.03
TEMPERATURE (AIR)       2.46
SPECIFIC CONDUCTANCE    3.52
LITHIUM TOTAL           5.91
VANADIUM TOTAL          6.00
BERYLLIUM TOTAL         6.00
BARIUM TOTAL            6.00
IRON TOTAL              6.02
MANGANESE TOTAL         6.07
ALUMINUM TOTAL          6.16
dtype: float64

Output saved to: E